Modeling data

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, classification_report, 
                             confusion_matrix, ConfusionMatrixDisplay)
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Loading dataset
df = pd.read_csv('data/processed/features.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:\n{df['churn_risk'].value_counts()}")

Dataset shape: (97007, 14)

Target distribution:
churn_risk
0    82098
1    14909
Name: count, dtype: int64


In [2]:
# Encoding product_category
le = LabelEncoder()
df['product_category_encoded'] = le.fit_transform(df['product_category'])

# Separating features and target
X = df.drop(columns=['churn_risk', 'product_category'])
y = df['churn_risk']

# Splitting into train and test (80/20, stratified to preserve target proportion)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape}")
print(f"Test size: {X_test.shape}")
print(f"\nTarget proportion in train: {y_train.mean()*100:.1f}%")
print(f"Target proportion in test: {y_test.mean()*100:.1f}%")

Train size: (77605, 13)
Test size: (19402, 13)

Target proportion in train: 15.4%
Target proportion in test: 15.4%


In [3]:
# Training models

# 1. Logistic Regression (baseline)
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_auc = roc_auc_score(y_test, lr.predict_proba(X_test)[:, 1])
print(f"Logistic Regression AUC-ROC: {lr_auc:.4f}")

# 2. Random Forest
rf = RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
print(f"Random Forest AUC-ROC:       {rf_auc:.4f}")

# 3. XGBoost
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]
xgb = XGBClassifier(scale_pos_weight=scale_pos_weight, 
                    n_estimators=100, random_state=42, eval_metric='auc')
xgb.fit(X_train, y_train)
xgb_auc = roc_auc_score(y_test, xgb.predict_proba(X_test)[:, 1])
print(f"XGBoost AUC-ROC:             {xgb_auc:.4f}")

Logistic Regression AUC-ROC: 0.6494
Random Forest AUC-ROC:       0.6612
XGBoost AUC-ROC:             0.6580
